# Durables with supplied age

This notebook constructs the model in four steps and checks its supplied terminal age. It displays selected stage objects and their methods; it does not solve the model.

## Recipe

The recipe supplies two fixed income tables, `Y` and `y_R`. Repeated stages read their rows at the age field.

In [1]:
from pathlib import Path

from bellman.trellis import (
    calibrate_trellis, configure_trellis, connection_report,
    elaborate_trellis, load_recipe, methodize_trellis,
)

recipe = Path("../declaration/spec.yml")
syntactic = load_recipe(recipe.parent, recipe.name)
syntactic["periods"][1]["stages"]["tenure"]["calibration"]

{'R': 1.045,
 'R_H': 1.0,
 'β': 0.945,
 'α': 0.7,
 'γ_c': 3.5,
 'γ_h': 1.5,
 'κ': 0.06,
 'δ': 0.0,
 'τ': 0.12,
 'ρ_z': 0.82,
 'σ_z': 0.11,
 'θ': 1.3498,
 'K': 1.3,
 'Y': [0.0,
  0.3193257731132868,
  0.35488542610341706,
  0.3898899399390923,
  0.4238637223478638,
  0.4564083412535708,
  0.48721295853095614,
  0.5160584808353706,
  0.542816211534675,
  0.5674420865477191,
  0.5899677025802004,
  0.6104893288347982,
  0.6291559719037418,
  0.6461573784552829,
  0.6617126471889255,
  0.6760599084014481,
  0.6894473357801711,
  0.7021255920034827,
  0.7143416819432543,
  0.7263340943388776,
  0.7383290511606643,
  0.7505376482295287,
  0.763153655230744,
  0.7763517425853104,
  0.7902859120804835,
  0.8050879241589475,
  0.8208655349560192,
  0.8377003793543313,
  0.8556453623796897,
  0.8747214510437913,
  0.8949137939366003,
  0.9161671388401348,
  0.9171372134145425,
  0.9182779237754403,
  0.9193630506447101,
  0.9201273588658764,
  0.9202669711794874,
  0.9194408320489058,
  0.917273

## Elaboration

Elaboration constructs the symbolic stages and their connections.

In [2]:
elaborated = elaborate_trellis(syntactic)
elaborated["periods"][1]["stages"]["tenure"]

ElaboratedStage('tenure', methods)

## Methodization

The recipe’s additions select the numerical methods. The explicit methodization step retains additions already read during elaboration.

In [3]:
methodized = methodize_trellis(elaborated, syntactic)
methodized["periods"][1]["stages"]["tenure"].methods

arvl_to_dcsn               !monte_carlo(n_paths=n_sim, seed=sim_seed)   "fues-durables:make_tenure_forward"
cntn_to_dcsn.evaluate[V]   !Linear(extrapolate=extrap_policy, clamp_factor=clamp_max_factor)   !Cartesian(a_d=(a_min, a_max, n_a), h_d=(h_min, h_max, n_h))   "dcsmm:interp_as_scalar"
dcsn_to_arvl.value.E_ε     !tauchen(n_income_states=N_wage, n_std=tauchen_n_std, min_prob=tauchen_min_prob)   "fues-durables:make_conditioners"
policy                     !max(tie_break=tie_break)   "fues-durables:make_tenure_ops"

## Calibration

Calibration binds parameter values, including the fixed income tables.

In [4]:
calibrated = calibrate_trellis(methodized, syntactic)
calibrated["periods"][1]["stages"]["tenure"].par

name  set       value                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     describes
R     ℝ₊₊       1.045                                                                               

## Settings

Configuration binds the numerical settings.

In [5]:
trellis = configure_trellis(calibrated, syntactic)
trellis["periods"][1]["stages"]["tenure"].settings

name                          set             value   read at                   by            grid
tie_break                     {keep, adjust}  adjust  policy                    !max
n_a                           ℕ               300     cntn_to_dcsn.evaluate[V]  !Linear       !Cartesian
a_min                         ℝ₊              0.01    cntn_to_dcsn.evaluate[V]  !Linear       !Cartesian
a_max                         ℝ₊              7.5     cntn_to_dcsn.evaluate[V]  !Linear       !Cartesian
n_h                           ℕ               300     cntn_to_dcsn.evaluate[V]  !Linear       !Cartesian
h_min                         ℝ₊₊             0.01    cntn_to_dcsn.evaluate[V]  !Linear       !Cartesian
h_max                         ℝ₊₊             10.0    cntn_to_dcsn.evaluate[V]  !Linear       !Cartesian
extrap_policy                 {0, 1}          0       cntn_to_dcsn.evaluate[V]  !Linear
clamp_max_factor              ℝ₊₊             1.1     cntn_to_dcsn.evaluate[V]  !Linear
N_wage    

## Terminal inputs and connections

The bequest keeps supplied age 71. Earlier tenure choices send back ages 70 through 20. There must be 51 periods and 204 certified routes. A callable continuation retains conditions that point checks alone cannot decide.

In [6]:
terminal = trellis["periods"][0]["stages"]["bequest"]
requirements = terminal.check_inputs({"age": 71, "V[>]": lambda x: 0.0})
routes = connection_report(trellis)
assert len(trellis["periods"]) == 51
assert routes.counts["certified"] == 204
print(len(trellis["periods"]), "periods,", routes.counts["certified"], "certified routes")
print(requirements)
periods_to_inspect = (0, 10, 50)

51 periods, 204 certified routes
(Condition(condition='continuation-input', entries=(Identity(kind='field', stage='bequest', address=None, perch='cntn', field='V', owner=None, subjects=(), occurrence=None, mark=None), Named(name='ℝ')), status='required', evidence=(), explanation='The representing function for V[>] must be measurable on its declared domain, depend only on () in that order, and take values in ℝ.'),)


## Selected stages

The following cells show the terminal period, the last working period and the first period.

In [7]:
# the printed sheet of each selected stage occurrence, under its position
for h in periods_to_inspect:
    period = trellis["periods"][h]
    for key, stage in period["stages"].items():
        print(f"periods[{h}] / {key}")
        print(stage.view(), end="\n\n")

periods[0] / tenure
BELLMAN-SYM / STAGE tenure                               [VERSION = SPECIFIED]
a stage occurrence of ../declaration/stages/tenure/tenure.bl
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

PERCHES ──────────────────────────────────────────────────────────────────────
  ≺ arrival        a≺ state, h≺ state, z≺ state, V≺ value, age≺ value
  ∼ decision       a_d state, h_d state, z_d state, d control, V value, age
                   value
  ≻ continuation   age_kp≻ value, age_adj≻ value, w_kp≻ state, h_kp≻ state,
                   z_kp≻ state, w_adj≻ state, z_adj≻ state, V_kp≻ value,
                   V_adj≻ value
  ≺ → ∼            ε drawn after arrival, before decision

FIELDS ───────────────────────────────────────────────────────────────────────
  states           a≺ at arrival, h≺ at arrival, z≺ at arrival, a_d at
                   decision, h_d at decision, z_d at decision, w_kp≻ at
                   continuation, h_kp≻ at contin

BELLMAN-SYM / STAGE tenure                               [VERSION = SPECIFIED]
a stage occurrence of ../declaration/stages/tenure/tenure.bl
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

PERCHES ──────────────────────────────────────────────────────────────────────
  ≺ arrival        a≺ state, h≺ state, z≺ state, V≺ value, age≺ value
  ∼ decision       a_d state, h_d state, z_d state, d control, V value, age
                   value
  ≻ continuation   age_kp≻ value, age_adj≻ value, w_kp≻ state, h_kp≻ state,
                   z_kp≻ state, w_adj≻ state, z_adj≻ state, V_kp≻ value,
                   V_adj≻ value
  ≺ → ∼            ε drawn after arrival, before decision

FIELDS ───────────────────────────────────────────────────────────────────────
  states           a≺ at arrival, h≺ at arrival, z≺ at arrival, a_d at
                   decision, h_d at decision, z_d at decision, w_kp≻ at
                   continuation, h_kp≻ at continuation, z_kp≻ at con

In [8]:
# the first seven characters of the SHA-256 digest of each stage
# occurrence's text and of each mapping it is bound with, read when this
# cell runs, as the status line of Bellman Workbench prints them
for h in periods_to_inspect:
    period = trellis["periods"][h]
    for key, stage in period["stages"].items():
        sources = stage.provenance["sources"]
        digests = [f"text {stage.provenance['sha256'][:7]}"]
        digests += [
            f"{kind} {sources[kind]['sha256'][:7]}"
            for kind in ("calibration", "methods", "settings")
            if kind in sources
        ]
        print(f"periods[{h}] / {key}", *digests, sep=" · ")

periods[0] / tenure · text 6a83d2e · calibration 270051b · methods 8923971 · settings 3eb0493
periods[0] / keeper_cons · text fc9742b · calibration 328c67c · methods ed02fae · settings 3eb0493
periods[0] / adjuster_cons · text 257bf55 · calibration 328c67c · methods 234f941 · settings 3eb0493
periods[0] / bequest · text b78488f · calibration 328c67c · methods 66445c9 · settings 2757122
periods[10] / tenure · text 6a83d2e · calibration 270051b · methods 8923971 · settings 3eb0493
periods[10] / keeper_cons · text fc9742b · calibration 328c67c · methods ed02fae · settings 3eb0493
periods[10] / adjuster_cons · text 257bf55 · calibration 328c67c · methods 234f941 · settings 3eb0493
periods[50] / tenure · text 6a83d2e · calibration 270051b · methods 8923971 · settings 3eb0493
periods[50] / keeper_cons · text fc9742b · calibration 328c67c · methods ed02fae · settings 3eb0493
periods[50] / adjuster_cons · text 257bf55 · calibration 328c67c · methods 234f941 · settings 3eb0493


In [9]:
# the methods bound to each selected stage occurrence
for h in periods_to_inspect:
    period = trellis["periods"][h]
    for key, stage in period["stages"].items():
        print(f"periods[{h}] / {key}")
        print(stage.methods, end="\n\n")

periods[0] / tenure
arvl_to_dcsn               !monte_carlo(n_paths=n_sim, seed=sim_seed)   "fues-durables:make_tenure_forward"
cntn_to_dcsn.evaluate[V]   !Linear(extrapolate=extrap_policy, clamp_factor=clamp_max_factor)   !Cartesian(a_d=(a_min, a_max, n_a), h_d=(h_min, h_max, n_h))   "dcsmm:interp_as_scalar"
dcsn_to_arvl.value.E_ε     !tauchen(n_income_states=N_wage, n_std=tauchen_n_std, min_prob=tauchen_min_prob)   "fues-durables:make_conditioners"
policy                     !max(tie_break=tie_break)   "fues-durables:make_tenure_ops"

periods[0] / keeper_cons
cntn_to_dcsn.evaluate   !Linear(extrapolate=extrap_policy, clamp_factor=clamp_max_factor)   !Cartesian(w_d=(a_min, a_max, n_a), h_d=(h_min, h_max, n_h))   "dcsmm:interp_as"
policy                  !egm(spacing=grid_phi, savings_floor=b, upper_envelope=keeper_upper_envelope, jump_threshold=m_bar, lookback=fues_lb, eps_d=fues_eps_d, eps_sep=fues_eps_sep, eps_fwd_back=fues_eps_fwd_back, parallel_guard=fues_parallel_guard, include_i

## Observations

- Terminal inputs and retained conditions:
- Income tables and age inputs:
- Decision after inspection: